## 1. Imports & Load Raw Data

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [43]:
df = pd.read_csv('dataset/avocado.csv')
y_target = 'AveragePrice'
df.head(5)

,Unnamed: 0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


## 2. Handle Missing / Invalid Values

In [44]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 0
Missing Feature:
[]


In [45]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    if missing_percentage > 5.0 :
        df = df.drop(columns=col)
    else:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [46]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [47]:
feature_numerik = df.select_dtypes(include=np.number).columns
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")


Jumlah Outlier Sebelum Dihapus: 6711
Jumlah Outlier Sesudah Dihapus: 0


## 5. Features Engineering

In [48]:
df.head()

,Unnamed: 0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


### 5.1 Rename Columns

In [49]:
df = df.rename(columns={'4046':'size_small','4225':'size_medium','4770':'size_extralarge'})

### 5.2 Extracing feature Date

In [50]:
df['Date'] = pd.to_datetime(df['Date'])
df['Tahun'] = df['Date'].dt.year
df['Bulan'] = df['Date'].dt.month
df['Hari'] = df['Date'].dt.day

### 5.3 Remove No Relevan Features

In [51]:
df= df.drop(columns=['Unnamed: 0'])

### 5.4 Create new Feature

In [52]:
df['ratio_small'] = np.where(df['Total Volume'] == 0, 0, df['size_small'] / df['Total Volume'])
df['ratio_medium'] = np.where(df['Total Volume'] == 0, 0, df['size_medium'] / df['Total Volume'])
df['ratio_bags'] = np.where(df['Total Volume'] == 0, 0, df['Total Bags'] / df['Total Volume'])
df['is_organic'] = np.where(df['type'] == 'organic', 'Ya', 'Bukan Organic')

## 6.Save Cleaned Dataset

In [55]:
file_path = 'dataset/avocado_Cleaning.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File sudah ada. Proses penyimpanan CSV dilewati (skip)


,Date,AveragePrice,Total Volume,size_small,size_medium,size_extralarge,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region,Tahun,Bulan,Hari,ratio_small,ratio_medium,ratio_bags,is_organic
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany,2015,12,27,0.016139,0.847723,0.135388,Bukan Organic
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany,2015,12,20,0.012287,0.813434,0.173216,Bukan Organic
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany,2015,12,13,0.006722,0.923274,0.068900,Bukan Organic
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany,2015,12,6,0.014331,0.911184,0.073566,Bukan Organic
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany,2015,11,29,0.018446,0.858909,0.121160,Bukan Organic
